# ② 데이터 로드 & 구조 확인 — Load & View
**11team · 서울 지하철 혼잡도 분석**

분석에 쓸 데이터를 불러오고 구조(크기·컬럼·샘플)를 확인한다.
본격 정제·분석은 다음 단계(③ EDA)에서 진행한다.

## 사용 데이터
| 데이터 | 내용 | 출처 |
|---|---|---|
| 지하철 승하차 | 호선·역별 시간대별 승하차 (2015–2021) | 서울 열린데이터광장 |
| 역 위치좌표 | 지하철역 위·경도 | Kakao Local API |
| 주야간 인구 | 자치구별 주간인구지수 (2020) | 공공데이터포털(통계청) |
| 직업유형별 취업인구 | 자치구별 직업군 취업인구 (2020) | 서울 열린데이터광장 |
| (기상) | 서울 월별 기온·강수 | 기상청 — *월별 재다운로드 후 추가* |

In [1]:
import pandas as pd, numpy as np

pd.set_option('display.max_columns', 60)

SUBWAY_DIR = r"C:/Users/최용우/Downloads/drive-download-20260625T013956Z-3-001/dataset"
EXT_DIR    = r"C:/Users/최용우/claude/yearsubway_11team/external_data"

def read_csv_auto(path, **kw):
    """인코딩 자동 감지 CSV 로더 (cp949/utf-8 혼용 대응)."""
    for enc in ['utf-8-sig', 'cp949', 'euc-kr', 'utf-8']:
        try:
            return pd.read_csv(path, encoding=enc, **kw)
        except UnicodeDecodeError:
            continue
    raise RuntimeError('인코딩 감지 실패: ' + path)

## 1. 지하철 승하차 인원 (메인 데이터)

In [2]:
metro = read_csv_auto(f"{SUBWAY_DIR}/Seoul_subway_data_20210705.csv")
print("shape:", metro.shape)
metro.head()

shape: (45338, 52)


,사용월,호선명,지하철역,04시-05시 승차인원,04시-05시 하차인원,05시-06시 승차인원,05시-06시 하차인원,06시-07시 승차인원,06시-07시 하차인원,07시-08시 승차인원,07시-08시 하차인원,08시-09시 승차인원,08시-09시 하차인원,09시-10시 승차인원,09시-10시 하차인원,10시-11시 승차인원,10시-11시 하차인원,11시-12시 승차인원,11시-12시 하차인원,12시-13시 승차인원,12시-13시 하차인원,13시-14시 승차인원,13시-14시 하차인원,14시-15시 승차인원,14시-15시 하차인원,15시-16시 승차인원,15시-16시 하차인원,16시-17시 승차인원,16시-17시 하차인원,17시-18시 승차인원,17시-18시 하차인원,18시-19시 승차인원,18시-19시 하차인원,19시-20시 승차인원,19시-20시 하차인원,20시-21시 승차인원,20시-21시 하차인원,21시-22시 승차인원,21시-22시 하차인원,22시-23시 승차인원,22시-23시 하차인원,23시-24시 승차인원,23시-24시 하차인원,00시-01시 승차인원,00시-01시 하차인원,01시-02시 승차인원,01시-02시 하차인원,02시-03시 승차인원,02시-03시 하차인원,03시-04시 승차인원,03시-04시 하차인원,작업일자
0,202106,1호선,동대문,715,14,13235,2131,8936,6979,14776,12395,18660,24732,16788,22866,15988,21388,17257,22109,20561,21732,21099,20786,22318,19616,23370,18703,24338,17325,23923,17672,22895,18354,15871,19089,12693,12789,13040,10134,11167,10601,2811,8211,16,1434,1,1,0,0,0,0,20210703
1,202106,1호선,동묘앞,51,1,3218,1100,3422,4802,5896,9703,9194,24921,8022,17333,9687,19292,14091,24305,20089,26186,24776,28141,27144,26643,28360,23213,31119,17744,27036,13759,23606,10098,11006,6510,6119,4409,5485,4265,3405,5689,1035,2589,4,1348,0,0,0,0,0,0,20210703
2,202106,1호선,서울역,654,17,9008,6400,12474,37203,37253,91875,59876,187805,44619,118679,42611,57710,49533,50003,59357,53317,61171,53687,53310,49094,65767,52788,76249,53969,122928,64693,184907,73978,87575,46769,59961,30743,65078,27435,44921,22829,11581,8024,30,637,0,1,0,0,0,0,20210703
3,202106,1호선,시청,37,0,1881,4340,2948,21443,6280,62346,7740,167991,8117,72853,9284,29250,14030,27989,15295,25037,18849,24492,23331,20032,30469,17869,36116,15593,66595,16611,135842,17805,46850,8139,38173,4455,39048,4234,28501,3686,4390,1485,3,92,0,0,0,0,0,0,20210703
4,202106,1호선,신설동,343,3,8150,3192,8131,10929,17021,25745,24583,62999,16472,33400,14689,20639,16427,17238,17625,16913,19712,17440,19363,16063,20684,16772,28784,17481,39357,20038,57642,25012,20823,17529,13880,10932,13378,10102,10189,11809,1952,5451,10,449,0,0,0,0,0,0,20210703


In [3]:
print("기간 :", metro['사용월'].min(), "~", metro['사용월'].max())
print("호선 :", metro['호선명'].nunique(), "개 / 역 :", metro['지하철역'].nunique(), "개")
print("결측 :", int(metro.isna().sum().sum()), "건")

기간 : 201501 ~ 202106
호선 : 26 개 / 역 : 579 개
결측 : 0 건


## 2. 역 위치좌표 (지도 시각화용)

In [4]:
subway_loc = read_csv_auto(f"{SUBWAY_DIR}/subway_location_data.csv")
print("shape:", subway_loc.shape)
subway_loc.head()

shape: (579, 4)


,지하철역,주소,x좌표,y좌표
0,4.19민주묘지역,서울 강북구 우이동 72-182,37.649457,127.013506
1,가능역,경기 의정부시 가능동 197-1,37.747906,127.044358
2,가락시장역,서울 송파구 가락동 184-23,37.492915,127.118215
3,가산디지털단지역,서울 금천구 가산동 468-4,37.482414,126.882240
4,가양역,서울 강서구 가양동 14-61,37.561758,126.853997


## 3. 주야간 인구 — 외부데이터 ①
**주간인구지수** = 주간인구 / 상주인구 × 100.
100 초과 → 낮에 사람이 모이는 **업무지구**, 100 미만 → **주거지(베드타운)**.
원본은 연령·성별로 쪼개져 있어 **자치구 합계** 행만 추출한다.

In [5]:
daynight_raw = pd.read_excel(
    f"{EXT_DIR}/서울특별시_자치구별 연령별 주간 야간 인구_20201231.xlsx", header=1)
daynight_raw['행정구역별'] = daynight_raw['행정구역별'].ffill()

# 성별=계 & 연령별=합계 & 서울특별시 전체 제외 → 자치구별 1행
daynight = daynight_raw[
    (daynight_raw['성별'] == '계') &
    (daynight_raw['연령별'] == '합계') &
    (daynight_raw['행정구역별'] != '서울특별시')].copy()
daynight['자치구'] = daynight['행정구역별'].str.replace('\u3000', '', regex=False).str.strip()
daynight = daynight[['자치구', '상주인구', '주간 인구', '주간 인구 지수']].reset_index(drop=True)

print("자치구 수:", len(daynight))
daynight.sort_values('주간 인구 지수', ascending=False).head(8)

자치구 수: 25


,자치구,상주인구,주간 인구,주간 인구 지수
1,중구,118450,380224,321.0
0,종로구,141316,345797,244.7
22,강남구,499028,961600,192.7
18,영등포구,360059,514216,142.8
21,서초구,394955,553702,140.2
17,금천구,224123,288203,128.6
2,용산구,212043,265108,125.0
13,마포구,354420,428016,120.8


## 4. 직업유형별 취업인구 — 외부데이터 ②

In [6]:
job_raw = read_csv_auto(f"{EXT_DIR}/직업유형별+취업인구(구별)_20260625143926.csv")
print("shape:", job_raw.shape)
job_raw.head(4)

shape: (29, 38)


,자치구별(1),자치구별(2),2020,2020.1,2020.2,2020.3,2020.4,2020.5,2020.6,2020.7,2020.8,2020.9,2020.10,2020.11,2020.12,2020.13,2020.14,2020.15,2020.16,2020.17,2020.18,2020.19,2020.20,2020.21,2020.22,2020.23,2020.24,2020.25,2020.26,2020.27,2020.28,2020.29,2020.30,2020.31,2020.32,2020.33,2020.34,2020.35
0,자치구별(1),자치구별(2),합계,합계,합계,관리자,관리자,관리자,전문가 및 관련종사자,전문가 및 관련종사자,전문가 및 관련종사자,사무종사자,사무종사자,사무종사자,서비스종사자,서비스종사자,서비스종사자,판매종사자,판매종사자,판매종사자,농림어업 숙련 종사자,농림어업 숙련 종사자,농림어업 숙련 종사자,기능원 및 관련기능종사자,기능원 및 관련기능종사자,기능원 및 관련기능종사자,장치기계조작 및 조립종사자,장치기계조작 및 조립종사자,장치기계조작 및 조립종사자,단순노무종사자,단순노무종사자,단순노무종사자,기타,기타,기타,미상,미상,미상
1,자치구별(1),자치구별(2),합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계
2,자치구별(1),자치구별(2),소계,남자,여자,소계,남자,여자,소계,남자,여자,소계,남자,여자,소계,남자,여자,소계,남자,여자,소계,남자,여자,소계,남자,여자,소계,남자,여자,소계,남자,여자,소계,남자,여자,소계,남자,여자
3,합계,소계,4922176,2679464,2242712,43390,34058,9332,1278200,673817,604383,1288948,635316,653632,596333,227990,368343,637653,329661,307992,8334,5985,2349,375483,293130,82353,267304,237880,29424,419370,235067,184303,7161,6560,601,-,-,-


## 5. 기상 — 외부데이터 ③ *(월별 재다운로드 후 추가 예정)*
처음 받은 파일이 1개월치(30일)만 들어와, **자료구분 '월'**로 다시 받아 합칠 예정.
Join 키 = `연-월` (지하철 사용월과 매칭).

---
## ✅ 로드 요약
- **지하철 승하차** (45,338 × 52) · **역 좌표** (579) — 메인
- **주야간 인구** (자치구 25) · **직업 취업인구** — 외부 Join용 (키: 자치구)
- **기상** — 월별 재다운로드 후 추가

### ▶ 다음 단계 (③ EDA)
시간대 패턴·호선·역 혼잡 분석 + **자치구 주간인구지수 Join**으로 방향성(업무지구 vs 베드타운) 검증.